# 01. Case Base Preparation & Preprocessing

This notebook performs the initial text extraction, cleaning, tokenization, stopword removal, and stemming for the document forgery (Pemalsuan Surat) dataset.

### Pipeline Steps:
1. **Read PDFs**: Load all raw PDF files from `data/raw_pdf/`.
2. **Text Extraction**: Extract page-by-page text using `pdfplumber` and merge them.
3. **Text Cleaning**:
   - Convert text to lowercase.
   - Remove headers, footers, watermarks, and page numbers using regex.
   - Remove punctuation and numbers.
4. **Tokenization**: Segment text into individual words.
5. **Stopword Removal**: Remove common Indonesian stopwords using `Sastrawi`.
6. **Stemming**: Apply `Sastrawi` stemming to reduce words to their base forms (lemma).
7. **Export**: Save the cleaned and processed text into `data/raw_text/` for downstream modeling.
8. **Logging**: Generate a preprocessing log in `results/cleaning_log.json` and a log file in `results/cleaning_log.log`.

In [ ]:
import os
import re
import json
import logging
import time
from pathlib import Path
import pdfplumber
import nltk
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
import pandas as pd

# Download NLTK resources
nltk.download('punkt', quiet=True)

In [ ]:
# Configure Logging
log_dir = Path("../results")
log_dir.mkdir(parents=True, exist_ok=True)
log_file = log_dir / "cleaning_log.log"

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(log_file, mode='w', encoding='utf-8'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger("CBR_Preprocessing")
logger.info("Preprocessing notebook initialized.")

In [ ]:
# Define paths
RAW_PDF_DIR = Path("../data/raw_pdf")
RAW_TEXT_DIR = Path("../data/raw_text")
PROCESSED_DIR = Path("../data/processed")

RAW_TEXT_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

logger.info(f"Raw PDF directory: {RAW_PDF_DIR.resolve()}")
logger.info(f"Target raw text directory: {RAW_TEXT_DIR.resolve()}")

In [ ]:
# Text Preprocessing & Cleaning Patterns

# Custom lists for Indonesian legal and administrative document watermarks, headers, and footers
WATERMARKS = [
    r'draft', r'confidential', r'rahasia', r'salinan', r'copy', r'asli',
    r'mahkamah agung', r'putusan', r'direktori putusan'
]

HEADER_FOOTER_PATTERNS = [
    r'^\s*halaman\s+\d+\s+dari\s+\d+\s*$',
    r'^\s*hal\.\s*\d+\s*$',
    r'^\s*page\s+\d+\s*$',
    r'^\s*mahkamah\s+agung\s+republik\s+indonesia\s*$',
    r'^\s*kepolisian\s+negara\s+republik\s+indonesia\s*$',
    r'^\s*kejaksaan\s+agung\s+republik\s+indonesia\s*$'
]

def clean_noise(text: str) -> str:
    """Removes watermarks, headers, footers, and page numbers."""
    lines = text.split('\n')
    cleaned_lines = []
    
    for line in lines:
        stripped = line.strip()
        # Check against header/footer patterns
        is_noise = False
        for pattern in HEADER_FOOTER_PATTERNS:
            if re.match(pattern, stripped, re.IGNORECASE):
                is_noise = True
                break
        if not is_noise:
            cleaned_lines.append(line)
            
    text = '\n'.join(cleaned_lines)
    
    # Remove watermarks inline
    for wm in WATERMARKS:
        text = re.sub(wm, '', text, flags=re.IGNORECASE)
        
    # Remove page numbers like "Halaman 1", "Page 2" anywhere
    text = re.sub(r'(?i)\b(?:halaman|hal\b|page|pg\.?)\s*\d+(?:\s*(?:dari|of)\s*\d+)?', '', text)
    
    # Remove single isolated page numbers on their own lines
    text = re.sub(r'^\s*\d+\s*$', '', text, flags=re.MULTILINE)
    
    return text

In [ ]:
# NLP Pipeline Components (Stopwords & Stemmer)

# Initialize Sastrawi components
stop_factory = StopWordRemoverFactory()
indonesian_stopwords = set(stop_factory.get_stop_words())

# Add custom Indonesian legal domain stopwords if necessary
custom_stopwords = {'dan', 'yang', 'untuk', 'dengan', 'pada', 'atau', 'adalah', 'bahwa'}
indonesian_stopwords.update(custom_stopwords)

stem_factory = StemmerFactory()
stemmer = stem_factory.create_stemmer()

def preprocess_text(raw_text: str) -> dict:
    """
    Executes full pipeline: lowercase, noise removal, punctuation removal,
    tokenization, stopword removal, and stemming.
    """
    # 1. Lowercase
    lowered = raw_text.lower()
    
    # 2. Clean Noise (Headers, footers, watermarks, page numbers)
    no_noise = clean_noise(lowered)
    
    # 3. Punctuation and Number Removal (keep only alphabetic characters)
    # Using regex to substitute non-letters with space
    clean_chars = re.sub(r'[^a-zA-Z\s]', ' ', no_noise)
    
    # 4. Tokenization
    tokens = nltk.word_tokenize(clean_chars)
    
    # 5. Stopword Removal
    filtered_tokens = [w for w in tokens if w not in indonesian_stopwords and len(w) > 1]
    
    # 6. Stemming using Sastrawi
    # Stemming word by word gives exact control.
    stemmed_tokens = []
    for token in filtered_tokens:
        stemmed = stemmer.stem(token)
        if len(stemmed) > 1:
            stemmed_tokens.append(stemmed)
            
    cleaned_text = ' '.join(stemmed_tokens)
    
    return {
        "original_char_count": len(raw_text),
        "cleaned_char_count": len(cleaned_text),
        "original_word_count": len(raw_text.split()),
        "cleaned_word_count": len(stemmed_tokens),
        "processed_text": cleaned_text
    }

In [ ]:
# Main Preprocessing Run

pdf_files = list(RAW_PDF_DIR.glob("*.pdf"))
cleaning_logs = []

logger.info(f"Found {len(pdf_files)} PDF files to process.")

if len(pdf_files) == 0:
    logger.warning("No PDF files found in data/raw_pdf/. Please place PDF files in that folder.")
else:
    for pdf_path in pdf_files:
        start_time = time.time()
        logger.info(f"Processing: {pdf_path.name}")
        
        try:
            # Extract text using pdfplumber
            full_text_list = []
            with pdfplumber.open(pdf_path) as pdf:
                for i, page in enumerate(pdf.pages):
                    page_text = page.extract_text()
                    if page_text:
                        full_text_list.append(page_text)
            
            merged_text = "\n".join(full_text_list)
            
            if not merged_text.strip():
                logger.error(f"Failed to extract text or empty text for {pdf_path.name}")
                cleaning_logs.append({
                    "filename": pdf_path.name,
                    "status": "failed",
                    "reason": "empty extracted text",
                    "duration_seconds": round(time.time() - start_time, 2)
                })
                continue
                
            # Preprocess text
            pipeline_result = preprocess_text(merged_text)
            
            # Save cleaned text
            txt_filename = pdf_path.stem + ".txt"
            output_path = RAW_TEXT_DIR / txt_filename
            with open(output_path, "w", encoding="utf-8") as f:
                f.write(pipeline_result["processed_text"])
                
            duration = round(time.time() - start_time, 2)
            logger.info(f"Successfully processed {pdf_path.name} in {duration}s. Words: {pipeline_result['original_word_count']} -> {pipeline_result['cleaned_word_count']}")
            
            cleaning_logs.append({
                "filename": pdf_path.name,
                "status": "success",
                "original_char_count": pipeline_result["original_char_count"],
                "cleaned_char_count": pipeline_result["cleaned_char_count"],
                "original_word_count": pipeline_result["original_word_count"],
                "cleaned_word_count": pipeline_result["cleaned_word_count"],
                "duration_seconds": duration
            })
            
        except Exception as e:
            logger.exception(f"Error processing {pdf_path.name}: {e}")
            cleaning_logs.append({
                "filename": pdf_path.name,
                "status": "error",
                "reason": str(e),
                "duration_seconds": round(time.time() - start_time, 2)
            })


In [ ]:
# Generate Preprocessing Cleaning Log Summary

summary_path = log_dir / "cleaning_log.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(cleaning_logs, f, indent=4)

logger.info(f"Cleaning log summary saved to {summary_path.resolve()}")

# Display log summary using pandas
if cleaning_logs:
    df_logs = pd.DataFrame(cleaning_logs)
    print("\nPreprocessing Run Summary:")
    display(df_logs)
else:
    print("\nNo files were processed.")